# Gemini API smoke test

Quick local sanity check before building the full benchmark harness. Confirms:
- the API key + SDK work end to end,
- `usage_metadata` actually returns input/output/total token counts for a multimodal (image) call,
- wall-clock latency measurement around the call works as expected.

Uses `google-genai` (the SDK the full harness will use) against `gemini-2.5-pro`, adapted from a Colab snippet for local/Jupyter use: API key comes from `GEMINI_API_KEY` in the environment (or a `.env` file), not `google.colab.userdata`, and there's no `!pip` magic tied to a Colab runtime.

In [ ]:
%pip install -q -U google-genai python-dotenv

In [ ]:
import mimetypes
import os
import time
from getpass import getpass

from dotenv import load_dotenv
from google import genai
from google.genai import types

# Loads a .env file if present (see .env.example); does nothing if it's missing.
load_dotenv()

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    API_KEY = getpass("Enter your GEMINI_API_KEY: ")
if not API_KEY:
    raise ValueError("No API key provided. Set GEMINI_API_KEY in your environment/.env or enter it when prompted.")

MODEL_NAME = os.environ.get("GEMINI_MODEL", "gemini-2.5-pro")

client = genai.Client(api_key=API_KEY)
print(f"Client ready. Using model: {MODEL_NAME}")

In [ ]:
def process_document_image(image_path: str, prompt: str = "Extract all text from this image. Provide the text in a clear, readable format."):
    """Sends one image to Gemini and reports extracted text, token usage, and latency.

    Mirrors what Approach A's Agent 1 and every Approach B agent will do in the
    full harness, so this is also a sanity check on that code path.
    """
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        raise ValueError(f"Could not determine mime type for {image_path}")

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    contents = [types.Part.from_bytes(data=image_bytes, mime_type=mime_type), prompt]

    start = time.perf_counter()
    try:
        response = client.models.generate_content(model=MODEL_NAME, contents=contents)
    except Exception as e:
        elapsed = time.perf_counter() - start
        print(f"An error occurred during content generation after {elapsed:.2f}s: {e}")
        return None, 0, 0, elapsed
    elapsed = time.perf_counter() - start

    extracted_text = response.text

    input_token_count = 0
    output_token_count = 0
    total_token_count = 0
    usage = getattr(response, "usage_metadata", None)
    if usage is not None:
        input_token_count = getattr(usage, "prompt_token_count", 0) or 0
        output_token_count = getattr(usage, "candidates_token_count", 0) or 0
        total_token_count = getattr(usage, "total_token_count", 0) or 0
    else:
        print("Warning: usage_metadata not available in the response. Token counts cannot be measured.")

    print(f"Latency: {elapsed:.2f}s")
    print(f"Input tokens: {input_token_count} | Output tokens: {output_token_count} | Total tokens: {total_token_count}")

    return extracted_text, input_token_count, output_token_count, elapsed

In [ ]:
image_file_path = "dc_data/dc1.png"

print(f"Processing image: {image_file_path}")
text, input_tokens, output_tokens, latency_seconds = process_document_image(image_file_path)

if text:
    print("\n--- Extracted Text ---")
    print(text)
    print(f"\nInput={input_tokens}, Output={output_tokens}, Latency={latency_seconds:.2f}s")
else:
    print("Text extraction failed.")

In [ ]:
# Second sample document, to confirm results aren't a fluke of one image.
image_file_path_2 = "dc_data/dc2.png"

print(f"Processing image: {image_file_path_2}")
text2, input_tokens2, output_tokens2, latency_seconds2 = process_document_image(image_file_path_2)

if text2:
    print("\n--- Extracted Text ---")
    print(text2)
    print(f"\nInput={input_tokens2}, Output={output_tokens2}, Latency={latency_seconds2:.2f}s")
else:
    print("Text extraction failed.")